# Conatus Phronesis — merge + quantização GGUF pra rodar no Ollama local

Funde o adapter LoRA do experimento de reasoning com o base
`Qwen/Qwen3-8B`, converte pra GGUF e quantiza (Q4_K_M por padrão).
Runtime: GPU (T4/L4) acelera o merge; a conversão/quantização em si roda em CPU.

**Pré-requisito**: secret `HF_TOKEN` no Colab (ícone de chave) com acesso ao repo
privado do adapter.


In [ ]:
# 1) Login HF
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

BASE_MODEL = "Qwen/Qwen3-8B"
ADAPTER_REPO = userdata.get("PHRONESIS_8B_ADAPTER_REPO")
if not ADAPTER_REPO:
    raise ValueError("Configure o secret PHRONESIS_8B_ADAPTER_REPO com o repo do adapter 8B")
MERGED_DIR = "/content/merged"
GGUF_F16 = "/content/model-f16.gguf"
QUANT_TYPE = "Q4_K_M"   # troque pra Q5_K_M se quiser mais qualidade (~1GB a mais)
GGUF_QUANT = f"/content/model-{QUANT_TYPE}.gguf"


In [ ]:
# 2) Deps pro merge
%pip install -q -U transformers accelerate peft safetensors


In [ ]:
# 3) Merge: baixa base + adapter, funde os pesos, salva em formato HF (safetensors)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Carregando base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cpu")

print("Carregando adapter (repo privado)...")
model = PeftModel.from_pretrained(base, ADAPTER_REPO)

print("Fundindo LoRA nos pesos do base...")
model = model.merge_and_unload()

print(f"Salvando modelo fundido em {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merge concluido.")


In [ ]:
# 4) llama.cpp: clona e instala os requirements do script de conversao
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%pip install -q -r /content/llama.cpp/requirements.txt


In [ ]:
# 5) Converte o modelo fundido (HF/safetensors) pra GGUF f16
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16


In [ ]:
# 6) Compila o llama.cpp (cmake) pra ter o binario de quantizacao
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4


In [ ]:
# 7) Quantiza o f16 pro tipo escolhido (Q4_K_M: aproximadamente 5GB pro 8B)
!/content/llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_QUANT} {QUANT_TYPE}

import os
size_gb = os.path.getsize(GGUF_QUANT) / (1024**3)
print(f"
GGUF quantizado: {GGUF_QUANT} ({size_gb:.2f} GB)")


In [ ]:
# 8) Baixa o arquivo final pro seu computador
from google.colab import files
files.download(GGUF_QUANT)


## 9) (opcional) Persistir o GGUF no HF Hub em vez de baixar

Util se a conexao cair no meio do download, ou pra reusar depois sem regerar.


In [ ]:
# Opcional: sobe o GGUF pro mesmo repo privado (evita perder se o download falhar)
from huggingface_hub import HfApi
HfApi().upload_file(
    path_or_fileobj=GGUF_QUANT,
    path_in_repo=f"gguf/model-{QUANT_TYPE}.gguf",
    repo_id=ADAPTER_REPO,
    repo_type="model",
)
print("GGUF tambem disponivel em:", f"https://huggingface.co/{ADAPTER_REPO}/blob/main/gguf/model-{QUANT_TYPE}.gguf")


## 10) Rodar no Ollama — no SEU computador, não no Colab

Com o arquivo `.gguf` baixado, crie um `Modelfile` ao lado dele:

```
FROM ./model-Q4_K_M.gguf

PARAMETER temperature 0
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 8192
```

O GGUF já carrega o chat template do Qwen3 embutido (inclusive o formato
`<tool_call>` que o Phronesis foi treinado pra usar), então o Ollama deve renderizar
tools automaticamente via `/api/chat` com o parâmetro `tools`. Depois:

```bash
ollama create phronesis-thinking-8b -f Modelfile
ollama run phronesis-thinking-8b
```

**Nota sobre decodificação**: validamos no HF que `do_sample=False` +
`no_repeat_ngram_size=8` era a configuração que funcionava melhor (sem loop, sem
variância). O llama.cpp/Ollama não tem exatamente `no_repeat_ngram_size` — o
equivalente mais próximo é `temperature 0` + `repeat_penalty` (ajustado acima para
1.1). Se aparecer loop de repetição em respostas longas (o modo de falha que vimos
no greedy puro), suba `repeat_penalty` pra ~1.3 ou adicione `PARAMETER repeat_last_n
256` no Modelfile.
